# IMDb 50K Movie Reviews – NLP Project

## Step 1: Dataset Download and Loading

### 1.1 Install required libraries
Run this once. `kagglehub` downloads the dataset, `pandas` loads it as a table, `numpy` handles numbers.

In [1]:
%pip install kagglehub pandas numpy

Note: you may need to restart the kernel to use updated packages.


### 1.2 Import libraries

In [2]:
import os
import kagglehub
import numpy as np
import pandas as pd

### 1.3 Download the dataset with KaggleHub
KaggleHub downloads the files to a cache folder on your computer and gives us the folder path.

In [3]:
path = kagglehub.dataset_download(
    "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"
)

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Bismillah\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1


### 1.4 Find the CSV file inside the downloaded folder
We list every file in the folder and pick the one ending in `.csv`.

In [4]:
print("Files in dataset folder:", os.listdir(path))

csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
csv_path = os.path.join(path, csv_files[0])

print("CSV file found:", csv_path)

Files in dataset folder: ['IMDB Dataset.csv']
CSV file found: C:\Users\Bismillah\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1\IMDB Dataset.csv


### 1.5 Load the CSV into a Pandas DataFrame
`df` is our original dataset. We will **never modify it**; later steps will work on a copy.

In [5]:
df = pd.read_csv(csv_path)

### 1.6 First five rows

In [6]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


### 1.7 Shape, column names, data types and missing values

In [7]:
print("Shape (rows, columns):", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values in each column:")
print(df.isnull().sum())

Shape (rows, columns): (50000, 2)

Column names: ['review', 'sentiment']

Data types:
review       str
sentiment    str
dtype: object

Missing values in each column:
review       0
sentiment    0
dtype: int64


### 1.8 Understand the `review` and `sentiment` columns
- **review**: the full text of a movie review written by a user. This is the text we will process.
- **sentiment**: the label for that review, either `positive` or `negative`.

In [8]:
print("Unique sentiment labels:", df["sentiment"].unique())
print("\nNumber of reviews per sentiment:")
print(df["sentiment"].value_counts())

Unique sentiment labels: <ArrowStringArray>
['positive', 'negative']
Length: 2, dtype: str

Number of reviews per sentiment:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [9]:
print("Example review:\n")
print(df["review"].iloc[0])
print("\nSentiment:", df["sentiment"].iloc[0])

Example review:

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show 

In [10]:
# Length of each review in characters (kept in a separate variable, not added to df)
review_lengths = df["review"].str.len()

print("Review length statistics (characters):")
print(review_lengths.describe())

Review length statistics (characters):
count    50000.000000
mean      1309.431020
std        989.728014
min         32.000000
25%        699.000000
50%        970.000000
75%       1590.250000
max      13704.000000
Name: review, dtype: float64


# 2.1 Import the unicodedata library

In [11]:
import unicodedata

In [12]:
examples = [
    "cafe\u0301",          # 'é' written as 'e' + a separate accent mark
    "\ufb01lm",             # 'fi' written as one special ligature character
    "\uff29\uff2d\uff24\uff42",  # 'IMDb' written in full-width letters
    "great\u00a0movie",    # non-breaking space between the words
    "Wait\u2026",           # '...' written as one single character
]

for text in examples:
    normalized_text = unicodedata.normalize("NFKC", text)
    print("Original  :", repr(text), "| length:", len(text))
    print("Normalized:", repr(normalized_text), "| length:", len(normalized_text))
    print()

Original  : 'café' | length: 5
Normalized: 'café' | length: 4

Original  : 'ﬁlm' | length: 3
Normalized: 'film' | length: 4

Original  : 'ＩＭＤｂ' | length: 4
Normalized: 'IMDb' | length: 4

Original  : 'great\xa0movie' | length: 11
Normalized: 'great movie' | length: 11

Original  : 'Wait…' | length: 5
Normalized: 'Wait...' | length: 7



In [13]:
word_1 = "caf\u00e9"    # 'é' as one character
word_2 = "cafe\u0301"   # 'e' + accent mark

print("word_1:", word_1, "| word_2:", word_2)
print("Equal before normalization?", word_1 == word_2)
print("Equal after normalization? ",
      unicodedata.normalize("NFKC", word_1) == unicodedata.normalize("NFKC", word_2))

word_1: café | word_2: café
Equal before normalization? False
Equal after normalization?  True


# 2.2 Make a working copy of the dataset

In [14]:
df_processed = df.copy()

# 2.3 Create the normalization function

In [15]:
def normalize_unicode(text):
    # Replace the accent sign used as an apostrophe (don´t -> don't)
    text = text.replace("\u00b4", "'")
    # Replace the sign used as a double quote (¨director¨ -> "director")
    text = text.replace("\u00a8", '"')
    # Apply NFKC Unicode normalization
    return unicodedata.normalize("NFKC", text)

# 2.4 Apply normalization to the reviews

In [16]:
df_processed["normalized_review"] = df_processed["review"].apply(normalize_unicode)

df_processed.head()

,review,sentiment,normalized_review
0,One of the other reviewers has mentioned that ...,positive,One of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,A wonderful little production. <br /><br />The...
2,I thought this was a wonderful way to spend ti...,positive,I thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,Basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"Petter Mattei's ""Love in the Time of Money"" is..."


In [17]:
changed = df_processed["review"] != df_processed["normalized_review"]

print("Reviews changed by normalization:", changed.sum(), "out of", len(df_processed))

Reviews changed by normalization: 223 out of 50000


In [18]:
from collections import Counter

changed_chars = Counter()

for review in df_processed.loc[changed, "review"]:
    for ch in review:
        if ord(ch) > 127 and normalize_unicode(ch) != ch:
            changed_chars[ch] += 1

print("Most common characters changed by normalization:\n")
for ch, count in changed_chars.most_common(15):
    print(repr(ch), "->", repr(normalize_unicode(ch)),
          "|", unicodedata.name(ch, "UNKNOWN"), "|", count, "times")

Most common characters changed by normalization:

'´' -> "'" | ACUTE ACCENT | 434 times
'¨' -> '"' | DIAERESIS | 64 times
'\xa0' -> ' ' | NO-BREAK SPACE | 44 times
'½' -> '1⁄2' | VULGAR FRACTION ONE HALF | 39 times
'…' -> '...' | HORIZONTAL ELLIPSIS | 16 times
'¾' -> '3⁄4' | VULGAR FRACTION THREE QUARTERS | 4 times
'º' -> 'o' | MASCULINE ORDINAL INDICATOR | 2 times
'，' -> ',' | FULLWIDTH COMMA | 2 times
'″' -> '′′' | DOUBLE PRIME | 1 times
'³' -> '3' | SUPERSCRIPT THREE | 1 times
'ª' -> 'a' | FEMININE ORDINAL INDICATOR | 1 times


In [19]:
def show_change(review, window=40):
    for i, ch in enumerate(review):
        if ord(ch) > 127 and normalize_unicode(ch) != ch:
            start = max(0, i - window)
            end = i + window
            part = review[start:end]
            print("Original  :", repr(part))
            print("Normalized:", repr(normalize_unicode(part)))
            return
    print("(The change is in a combined character, for example 'e' + accent mark.)")


example_reviews = df_processed.loc[changed, "review"].head(3)

for number, review in enumerate(example_reviews, start=1):
    print("Example", number)
    show_change(review)
    print()

Example 1
Original  : 'lms. But, after seeing this film I haven´t. This is a unique masterpiece made by'
Normalized: "lms. But, after seeing this film I haven't. This is a unique masterpiece made by"

Example 2
Original  : 'a commercial from beginning to end. The ¨director¨spent the whole time on a trip'
Normalized: 'a commercial from beginning to end. The "director"spent the whole time on a trip'

Example 3
Original  : 'The poet Carne disappears (didn´t he disappeared with Prévert?) and is '
Normalized: "The poet Carne disappears (didn't he disappeared with Prévert?) and is "



# 2.5 confirmed that the original dataset is unchanged

In [20]:
print("Original df columns:   ", df.columns.tolist())
print("df_processed columns:  ", df_processed.columns.tolist())
print("Original review column unchanged:", df["review"].equals(df_processed["review"]))

Original df columns:    ['review', 'sentiment']
df_processed columns:   ['review', 'sentiment', 'normalized_review']
Original review column unchanged: True


# 3.1 Text Cleaning

In [21]:
import re
import html

In [22]:
html_tag_pattern = r"(?i)<(?:/?(?:p|i|b|u|em|strong)|br|hr)\s*/?\s*>|<a\s+href[^<>]*>|</a\s*>"

reviews_with_tags = df_processed["normalized_review"].str.contains(html_tag_pattern, regex=True)
print("Reviews containing HTML tags:", reviews_with_tags.sum(), "out of", len(df_processed))

tag_counts = Counter()
for review in df_processed.loc[reviews_with_tags, "normalized_review"]:
    for tag in re.findall(html_tag_pattern, review):
        tag_counts[tag.lower()] += 1

print("\nMost common HTML tags:")
for tag, count in tag_counts.most_common(10):
    print(repr(tag), "|", count, "times")

Reviews containing HTML tags: 29201 out of 50000

Most common HTML tags:
'<br />' | 201948 times
'<i>' | 8 times
'</i>' | 8 times
'<p>' | 1 times
'<hr>' | 1 times
'<em>' | 1 times
'</em>' | 1 times


In [23]:
control_char_pattern = r"[\x00-\x1f\x7f-\x9f]"
invisible_char_pattern = r"[\u00ad\u200b-\u200f\u202a-\u202e\u2060-\u2064\ufeff]"

unwanted_counts = Counter()
for review in df_processed["normalized_review"]:
    for ch in re.findall(control_char_pattern + "|" + invisible_char_pattern, review):
        unwanted_counts[ch] += 1

print("Unwanted characters found:\n")
for ch, count in unwanted_counts.most_common(15):
    print(repr(ch), "|", unicodedata.name(ch, "CONTROL CHARACTER"), "|", count, "times")

Unwanted characters found:

'\x96' | CONTROL CHARACTER | 2529 times
'\x85' | CONTROL CHARACTER | 1479 times
'\x97' | CONTROL CHARACTER | 596 times
'\x91' | CONTROL CHARACTER | 121 times
'\t' | CONTROL CHARACTER | 106 times
'\x84' | CONTROL CHARACTER | 31 times
'\x8e' | CONTROL CHARACTER | 18 times
'\x9e' | CONTROL CHARACTER | 18 times
'\x95' | CONTROL CHARACTER | 17 times
'\x80' | CONTROL CHARACTER | 8 times
'\xad' | SOFT HYPHEN | 5 times
'\x08' | CONTROL CHARACTER | 4 times
'\x8d' | CONTROL CHARACTER | 3 times
'\x10' | CONTROL CHARACTER | 1 times
'\x9a' | CONTROL CHARACTER | 1 times


# 3.2 Cleaning functions

In [24]:
# Broken characters from old Windows encoding -> what the writer meant
# Broken characters from old Windows encoding -> what the writer meant
broken_character_fixes = {
    "\x80": "€",     # euro sign
    "\x84": '"',     # low double quote
    "\x85": "...",   # ellipsis
    "\x8a": "Š",     # letter S with caron (in names)
    "\x8e": "Ž",     # letter Z with caron (in names)
    "\x91": "'",     # left single quote
    "\x92": "'",     # right single quote / apostrophe
    "\x93": '"',     # left double quote
    "\x94": '"',     # right double quote
    "\x96": "-",     # en dash
    "\x97": "-",     # em dash
    "\x9a": "š",     # letter s with caron (in names)
    "\x9e": "ž",     # letter z with caron (in names)
}

def remove_html_tags(text):
    return re.sub(html_tag_pattern, " ", text)


def fix_html_entities(text):
    return html.unescape(text)


def fix_unwanted_characters(text):
    for broken, fixed in broken_character_fixes.items():
        text = text.replace(broken, fixed)
    text = re.sub(control_char_pattern, " ", text)
    text = re.sub(invisible_char_pattern, "", text)
    return text


def remove_extra_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()


def clean_text(text):
    text = remove_html_tags(text)
    text = fix_html_entities(text)
    text = fix_unwanted_characters(text)
    text = remove_extra_whitespace(text)
    return text

# 3.3 Testing of cleaning function

In [25]:
test_text = "Great movie!<br /><br />The   acting was good \x96 not perfect.\x85 Tom &amp; Jerry   "

print("Before:", repr(test_text))
print("After: ", repr(clean_text(test_text)))

Before: 'Great movie!<br /><br />The   acting was good \x96 not perfect.\x85 Tom &amp; Jerry   '
After:  'Great movie! The acting was good - not perfect.... Tom & Jerry'


In [26]:
df_processed["cleaned_review"] = df_processed["normalized_review"].apply(clean_text)

df_processed.head()

,review,sentiment,normalized_review,cleaned_review
0,One of the other reviewers has mentioned that ...,positive,One of the other reviewers has mentioned that ...,One of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,A wonderful little production. <br /><br />The...,A wonderful little production. The filming tec...
2,I thought this was a wonderful way to spend ti...,positive,I thought this was a wonderful way to spend ti...,I thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,Basically there's a family where a little boy ...,Basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"Petter Mattei's ""Love in the Time of Money"" is...","Petter Mattei's ""Love in the Time of Money"" is..."


In [27]:
def show_before_after(index, n_chars=300):
    print("Review index:", index)
    print("BEFORE:", df_processed.loc[index, "normalized_review"][:n_chars])
    print()
    print("AFTER: ", df_processed.loc[index, "cleaned_review"][:n_chars])
    print("-" * 80)


def show_around(pattern, title, window=60):
    # Find the first review that matches the pattern, and show the text around the match
    for index, review in df_processed["normalized_review"].items():
        match = re.search(pattern, review)
        if match:
            start = max(0, match.start() - window)
            end = match.end() + window
            part = review[start:end]
            print(title)
            print("Review index:", index)
            print("BEFORE:", repr(part))
            print("AFTER: ", repr(clean_text(part)))
            print("-" * 80)
            return
    print(title, "-> no review found")


# Example 1: a review with <br /> tags
show_before_after(1)

# Example 2: a broken old-Windows character (dash)
show_around("\x96", "Example 2: broken dash character")

# Example 3: a broken old-Windows character (ellipsis)
show_around("\x85", "Example 3: broken ellipsis character")

# Example 4: an HTML entity such as &amp;
show_around(r"&(?:amp|quot|lt|gt|#\d+);", "Example 4: HTML entity")

# Example 5: the reviewer's own < > text is kept, not deleted
show_around(r"(?i)< cough >", "Example 5: reviewer's own < > text")

Review index: 1
BEFORE: A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" 

AFTER:  A wonderful little production. The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voice
--------------------------------------------------------------------------------
Example 2: broken dash character
Review index: 140
BEFORE: 'Shootout" (which seems like a poor name for a killing spree \x96 Meeker kills three people while trying to rob a convenience'
AFTER:  'Shootout" (which seems like a poor name for a killing spree - Meeker kills three people w

In [28]:
cleaned = df_processed["cleaned_review"]

print("Reviews still containing HTML tags:      ", cleaned.str.contains(html_tag_pattern, regex=True).sum())
print("Reviews still containing control chars:  ", cleaned.str.contains(control_char_pattern, regex=True).sum())
print("Reviews still containing double spaces:  ", cleaned.str.contains("  ", regex=False).sum())
print("Empty reviews after cleaning:            ", (cleaned.str.len() == 0).sum())

print("\nAverage length before cleaning:", round(df_processed["normalized_review"].str.len().mean(), 1), "characters")
print("Average length after cleaning: ", round(cleaned.str.len().mean(), 1), "characters")

Reviews still containing HTML tags:       0
Reviews still containing control chars:   0
Reviews still containing double spaces:   0
Empty reviews after cleaning:             0

Average length before cleaning: 1309.4 characters
Average length after cleaning:  1287.0 characters


In [29]:
print("Original df columns:   ", df.columns.tolist())
print("df_processed columns:  ", df_processed.columns.tolist())
print("Original review column unchanged:", df["review"].equals(df_processed["review"]))

Original df columns:    ['review', 'sentiment']
df_processed columns:   ['review', 'sentiment', 'normalized_review', 'cleaned_review']
Original review column unchanged: True
